[Microservices Architecture](https://www.atlassian.com/microservices/microservices-architecture)

Microservices architecture breaks down applications into small, independent services that communicate through well-defined APIs.



# Monolith vs Microservices

**Monolithic Architecture:**
```text
┌─────────────────────────────────┐
│     Single Application          │
│  ┌────────────────────────────┐ │
│  │ Weather Service            │ │
│  │ User Management            │ │
│  │ Notification Service       │ │
│  │ Shared Database            │ │
│  └────────────────────────────┘ │
└─────────────────────────────────┘
```

**Microservices Architecture:**
```text
┌──────────────┐  ┌──────────────┐  ┌──────────────┐
│   Weather    │  │    User      │  │ Notification │
│   Service    │  │  Management  │  │   Service    │
├──────────────┤  ├──────────────┤  ├──────────────┤
│ Weather DB   │  │  User DB     │  │  Message DB  │
└──────────────┘  └──────────────┘  └──────────────┘
       ↓                 ↓                   ↓
       └─────────── API Gateway ─────────────┘
```

<img src='./pic/monolith-microservices.png' width=500>

# Microservices Characteristics

1. **Independent deployment**: Each service can be deployed separately
2. **Technology diversity**: Different services can use different tech stacks
3. **Organized around business capabilities**: Each service represents a domain
4. **Decentralized data management**: Each service owns its database
5. **Fault isolation**: Failure in one service doesn't crash entire system
6. **Scalability**: Scale individual services based on demand



# Service Communication

## Synchronous Communication (REST/HTTP)

```python
    # Weather Service API
    @app.get("/weather/{city}")
    def get_weather(city: str):
        return {"city": city, "temperature": 20}

    # Client calling Weather Service
    import requests
    response = requests.get(f"http://weather-service/weather/{city}")
    weather = response.json()
```

**Advantages:**
- Simple and straightforward
- Immediate response
- Easy to debug

**Disadvantages:**
- Tight coupling
- Service dependency
- Cascading failures possible

## Asynchronous Communication (Message Queues)

```python
    # Producer (Weather Service)
    from celery import Celery
    celery = Celery('tasks', broker='redis://localhost:6379')

    @celery.task
    def process_weather_data(city):
        # Process weather data
        return {"city": city, "status": "processed"}

    # Consumer (Notification Service)
    @celery.task
    def send_weather_alert(weather_data):
        # Send notification based on weather
        pass
```

**Advantages:**
- Loose coupling
- Better fault tolerance
- Natural load balancing

**Disadvantages:**
- Eventual consistency
- More complex debugging
- Message ordering challenges



# API Gateway Pattern

API Gateway acts as single entry point for all clients. It is a server (or managed service) that:
- Sits between clients and backend services
- Accepts all incoming API requests
- Routes them to the right service
- Applies cross-cutting concerns consistently

**Responsibilities:**
- Request routing
- Authentication/Authorization
- Rate limiting
- Load balancing
- Response transformation
- Caching

<img src='./pic/API_gateway.gif' width=500>

**Example with FastAPI:**
```python
    # api_gateway.py
    from fastapi import FastAPI, HTTPException
    import httpx

    app = FastAPI()

    WEATHER_SERVICE = "http://weather-service:8000"
    USER_SERVICE = "http://user-service:8001"

    @app.get("/api/weather/{city}")
    async def get_weather(city: str, user_id: str):
        # Verify user
        async with httpx.AsyncClient() as client:
            user_resp = await client.get(f"{USER_SERVICE}/users/{user_id}")
            if user_resp.status_code != 200:
                raise HTTPException(401, "Unauthorized")
            
            # Get weather data
            weather_resp = await client.get(f"{WEATHER_SERVICE}/weather/{city}")
            return weather_resp.json()
```



## Common API Gateway implementations

You’ll see these a lot:
- **Cloud-managed**
  - AWS API Gateway
  - Azure API Management
  - Google Cloud API Gateway / Apigee
- **Self-hosted**
  - Kong
  - NGINX
  - Envoy
  - Spring Cloud Gateway

(Envoy is especially popular in gRPC + microservices setups.)

## API Gateway vs Service Mesh
They solve different problems and are usually used together, not instead of each other.
- API Gateway handles traffic into your system.
- Service Mesh handles traffic inside your system.

```text
       External Clients
   (Web / Mobile / Partner)
               ↓
            API Gateway (auth, rate limits, REST)
               ↓
    ┌─────────────────────────────────────────────────┐
    │   Service Mesh (mTLS, retries, traffic shaping) │
    │  (internal traffic)                             │
    │  Microservices A ↔ B ↔ C ↔ D (often gRPC)       │
    └─────────────────────────────────────────────────┘
```

### When you might use only one

Only API Gateway
- Monolith
- Few services
- Low internal complexity

Only Service Mesh
- Internal platform
- No external clients
- Batch jobs or internal tooling

Both (most production systems)
- Public-facing product
- Microservices at scale
- Compliance + observability needs

# Service Discovery

Services need to find each other dynamically.

**Solutions:**
- **Consul**: Service mesh with health checking
- **Eureka**: Netflix's service registry
- **Kubernetes DNS**: Built-in service discovery
- **Environment variables**: Simple for small setups



# Database per Service Pattern

Each microservice owns and manages its own database.

```text
    Weather Service → PostgreSQL (weather_logs)
    User Service → PostgreSQL (users, sessions)
    Notification Service → Redis (message queue)
    Analytics Service → MongoDB (aggregated data)
```

**Advantages:**
- Service independence
- Technology flexibility
- Scalability

**Challenges:**
- Distributed transactions
- Data consistency
- Querying across services



# Handling Distributed Transactions

## Saga Pattern

Break transaction into sequence of local transactions.

```python
# Order Saga Example
class OrderSaga:
    def create_order(self, order_data):
        try:
            # Step 1: Reserve inventory
            inventory_id = self.inventory_service.reserve(order_data)
            
            # Step 2: Process payment
            payment_id = self.payment_service.charge(order_data)
            
            # Step 3: Create order
            order_id = self.order_service.create(order_data)
            
            return {"order_id": order_id}
        except Exception as e:
            # Compensating transactions
            self.inventory_service.release(inventory_id)
            self.payment_service.refund(payment_id)
            raise
```



# Circuit Breaker Pattern

- Prevent cascading failures by detecting and handling failures.  
- A circuit breaker monitors failures when calling a dependency and temporarily blocks requests when failure thresholds are exceeded, allowing the system to fail fast and recover safely

**Without a circuit breaker ❌**:
- Service A keeps calling failing Service B
- Requests pile up
- Threads are exhausted
- Failures cascade → system outage

**With a circuit breaker ✅:**
- Fail fast
- Protect resources
- Recover gracefully

```python
from circuitbreaker import circuit

@circuit(failure_threshold=5, recovery_timeout=60)
def call_weather_service(city):
    response = requests.get(f"http://weather-service/weather/{city}")
    response.raise_for_status()
    return response.json()

# If 5 failures occur, circuit opens
# Requests fail fast for 60 seconds
# Then attempts recovery
```

**States**:  
```text
    CLOSED → OPEN → HALF-OPEN → CLOSED
```
1️⃣ Closed (normal)
- Requests flow normally
- Failures are counted

2️⃣ Open (tripped)
- Failure threshold exceeded
- Calls are blocked immediately
- Requests fail fast (fallback or error)

3️⃣ Half-Open (probe)
- After a cool-down period
- Allow a limited number of test requests
- If they succeed → close the circuit
- If they fail → open again

```text
    Service A
        |
        v
    [CLOSED] -- failures --> [OPEN]
        ^                       |
        |---- success ------[HALF-OPEN]
```

**Circuit breaker vs retry (important)**    
| Retry| Circuit Breaker| 
| ----| ----------------| 
| Tries again| Stops trying| 
| Good for transient issues| Good for persistent failures| 
| Can increase load| Reduces load| 

Best practice: **retry first, then circuit breaker**.

**Circuit breaker vs timeout**    
- Timeout: limits how long you wait
- Circuit breaker: decides whether to call at all

They are **complementary, not replacements**.

**How they work together**:  
```text
    Request
    ↓
    Timeout (limit wait time)
    ↓
    Failure recorded
    ↓
    Circuit breaker evaluates failure rate
    ↓
    Circuit opens → fail fast
```

# Monitoring and Observability

## Distributed Tracing
Track requests across multiple services.

```python
# Using OpenTelemetry
from opentelemetry import trace

tracer = trace.get_tracer(__name__)

@app.get("/weather/{city}")
def get_weather(city: str):
    with tracer.start_as_current_span("get_weather"):
        with tracer.start_as_current_span("call_external_api"):
            result = api_client.get_current_weather(city)
        
        with tracer.start_as_current_span("save_to_db"):
            db.add(WeatherLog(**result))
            db.commit()
        
        return result
```



## Centralized Logging
Aggregate logs from all services.

```python
import logging
import json

logger = logging.getLogger(__name__)

def log_request(request_id, service_name, action):
    logger.info(json.dumps({
        "request_id": request_id,
        "service": service_name,
        "action": action,
        "timestamp": datetime.now().isoformat()
    }))
```

**Tools:**
- **ELK Stack**: Elasticsearch, Logstash, Kibana
- **Grafana Loki**: Log aggregation
- **CloudWatch**: AWS logging solution

#### Health Checks

```python
@app.get("/health")
def health_check():
    return {
        "status": "healthy",
        "database": check_database_connection(),
        "cache": check_redis_connection(),
        "external_api": check_api_availability()
    }
```



# Container Orchestration



## Docker Compose (Development)

```yaml
# docker-compose.yml
version: '3.8'

services:
  weather-service:
    build: ./weather-service
    ports:
      - "8000:8000"
    environment:
      - DATABASE_URL=postgresql://db:5432/weather
      - REDIS_URL=redis://redis:6379
    depends_on:
      - db
      - redis
  
  user-service:
    build: ./user-service
    ports:
      - "8001:8000"
    environment:
      - DATABASE_URL=postgresql://db:5432/users
  
  db:
    image: postgres:15
    environment:
      - POSTGRES_PASSWORD=secret
  
  redis:
    image: redis:7
```



## Kubernetes (Production)

Kubernetes (K8s) is an **open-source container orchestration platform** that automates deploying, scaling, and managing containerized applications.  

<img src='./pic/kubernetes-architecture.jpg' width=450>

| Docker| Kubernetes| 
| ------| ----------| 
| Builds & runs containers| Manages many containers| 
| Single machine| Cluster of machines| 
| Low-level| Orchestration layer| 

- Docker ≠ Kubernetes
- Kubernetes uses container runtimes (Docker, containerd)

Where Kubernetes is used
- Microservices
- Data pipelines
- ML workloads
- APIs
- Cloud-native platforms

Managed versions:
- EKS (AWS)
- GKE (Google)
- AKS (Azure)


```yaml
# weather-service-deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: weather-service
spec:
  replicas: 3
  selector:
    matchLabels:
      app: weather-service
  template:
    metadata:
      labels:
        app: weather-service
    spec:
      containers:
      - name: weather-service
        image: weather-service:latest
        ports:
        - containerPort: 8000
        env:
        - name: DATABASE_URL
          valueFrom:
            secretKeyRef:
              name: weather-secrets
              key: database-url
 
apiVersion: v1
kind: Service
metadata:
  name: weather-service
spec:
  selector:
    app: weather-service
  ports:
  - port: 80
    targetPort: 8000
  type: LoadBalancer
```



# Microservices Best Practices

1. **Start with monolith**: Don't begin with microservices
2. **Define clear boundaries**: Based on business domains
3. **API versioning**: Support backward compatibility
4. **Automated testing**: Essential for distributed systems
5. **Monitoring**: Observability from day one
6. **Documentation**: Clear API contracts
7. **Security**: Authentication at gateway, authorization per service
8. **Data consistency**: Plan for eventual consistency
9. **Deployment automation**: CI/CD is mandatory
10. **Team structure**: Align teams with services (Conway's Law)

## When to Use Microservices

**Good fit:**
- Large, complex applications
- Multiple teams working independently
- Different scalability requirements
- Need for technology diversity
- Frequent deployments

**Not a good fit:**
- Small applications
- Simple CRUD operations
- Single small team
- Limited DevOps capability
- Tight latency requirements

## Example: Weather Application as Microservices

```text
┌─────────────────────┐
│    API Gateway      │
│   (FastAPI/Nginx)   │
└──────────┬──────────┘
           │
    ┌──────┴──────┬──────────┬────────────┐
    ↓             ↓          ↓            ↓
┌─────────┐  ┌────────┐  ┌────────┐  ┌────────┐
│ Weather │  │  User  │  │History │  │ Alert  │
│ Service │  │Service │  │Service │  │Service │
└────┬────┘  └───┬────┘  └───┬────┘  └───┬────┘
     │           │           │           │
┌────▼────┐  ┌───▼────┐  ┌───▼────┐  ┌───▼────┐
│Weather  │  │ User   │  │History │  │Message │
│   DB    │  │   DB   │  │   DB   │  │ Queue  │
└─────────┘  └────────┘  └────────┘  └────────┘
```

**Service Responsibilities:**

- **Weather Service**: Fetch current weather from external API
- **User Service**: Authentication, authorization, user profiles
- **History Service**: Store and retrieve historical weather data
- **Alert Service**: Send notifications based on weather conditions

 



# Integration: CI/CD + Agile + Microservices

## Complete Development Workflow

1. **Planning (Agile)**
   - Product Owner prioritizes user stories
   - Team estimates during sprint planning
   - Stories added to Kanban board

2. **Development**
   - Developer picks story from "To Do"
   - Creates feature branch
   - Develops microservice changes
   - Writes tests (TDD)
   - Moves card to "In Progress"

3. **Code Review**
   - Opens pull request
   - CI pipeline runs automatically
   - GitHub Actions executes tests
   - Code review by peers
   - Moves card to "Review"

4. **Testing**
   - Automated tests in CI
   - Manual QA if needed
   - Integration tests across services
   - Moves card to "Testing"

5. **Deployment (CD)**
   - Merge to main branch
   - Automated deployment pipeline
   - Deploy to staging
   - Run smoke tests
   - Deploy to production
   - Moves card to "Done"

6. **Monitoring**
   - Track metrics in production
   - Monitor service health
   - Gather user feedback
   - Plan next iteration

### GitHub Actions for Microservices

```yaml
name: Microservices CI/CD

on:
  push:
    branches: [main]
    paths:
      - 'services/weather/**'

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
    - uses: actions/checkout@v3
    
    - name: Run tests
      run: |
        cd services/weather
        pytest --cov
    
  build:
    needs: test
    runs-on: ubuntu-latest
    steps:
    - name: Build Docker image
      run: |
        docker build -t weather-service:${{ github.sha }} .
    
    - name: Push to registry
      run: |
        docker push weather-service:${{ github.sha }}
  
  deploy:
    needs: build
    runs-on: ubuntu-latest
    steps:
    - name: Deploy to Kubernetes
      run: |
        kubectl set image deployment/weather-service \
          weather-service=weather-service:${{ github.sha }}
```

 

